In [14]:
pip install --upgrade gradio

Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install --upgrade gradio --no-cache-dir

Note: you may need to restart the kernel to use updated packages.


In [18]:
pip show gradio

Name: gradio
Version: 6.10.0
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License-Expression: Apache-2.0
Location: c:\Users\admin\anaconda3\Lib\site-packages
Requires: aiofiles, anyio, audioop-lts, brotli, fastapi, ffmpy, gradio-client, groovy, hf-gradio, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pytz, pyyaml, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 
Note: you may

In [19]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import gradio as gr

# ==============================
# Load API key
# ==============================
load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

# ==============================
# Dummy Hospital Database
# ==============================
patients = {

    "P101": {
        "name": "Amit",
        "appointment": "2 April, 10:30 AM - Cardiology",
        "test_results": "Blood Test Normal | BP 120/80"
    },

    "P102": {
        "name": "Neha",
        "appointment": "5 April, 3:00 PM - Dermatology",
        "test_results": "Vitamin D Low | Allergy Mild"
    }

}

# ==============================
# Tools
# ==============================
def get_appointment(pid):

    if pid in patients:
        return f"📅 Appointment: {patients[pid]['appointment']}"

    return "❌ Patient not found"


def get_test_results(pid):

    if pid in patients:
        return f"🧪 Test Results: {patients[pid]['test_results']}"

    return "❌ Patient not found"


# ==============================
# LLM decides tool
# ==============================
def decide_action(query):

    prompt = f"""
    choose one function:

    get_appointment
    get_test_results

    appointment question → get_appointment
    report/test question → get_test_results

    query: {query}

    return only function name
    """

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content.lower()


# ==============================
# MCP Agent
# ==============================
def hospital_agent(message, patient_id, history):

    if history is None:
        history = []

    if not patient_id:

        history.append({
            "role":"assistant",
            "content":"⚠️ enter patient id"
        })

        return "", history


    action = decide_action(message)


    if "appointment" in action:

        response = get_appointment(patient_id)

    elif "test" in action:

        response = get_test_results(patient_id)

    else:

        response = "ask about appointment or test report"


    history.append({
        "role":"user",
        "content":message
    })

    history.append({
        "role":"assistant",
        "content":response
    })


    return "", history


# ==============================
# Gradio UI
# ==============================
with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Hospital Smart Assistant (MCP + NVIDIA)")


    patient_id = gr.Textbox(
        label="Patient ID",
        placeholder="P101"
    )


    chatbot = gr.Chatbot(height=400)


    msg = gr.Textbox(
        label="Ask Question"
    )


    state = gr.State([])


    msg.submit(

        hospital_agent,

        inputs=[msg, patient_id, state],

        outputs=[msg, chatbot]

    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
